# Notebook 7 — Radiology Report App
## Hospital-Grade UI · PDF Download · Full Pipeline

**Features:**
- Upload X-ray → get full radiology report
- Grad-CAM heatmap overlay
- Disease confidence bars
- Download professional PDF (hospital letterhead)
- All models loaded from saved checkpoints


## Cell 1 — Install Dependencies

In [1]:
import subprocess, sys

pkgs = ['flask', 'flask-cors', 'reportlab', 'Pillow', 'rouge-score']
for pkg in pkgs:
    try:
        __import__(pkg.replace('-','_').split('==')[0])
        print(f' {pkg} already installed')
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        print(f' {pkg} installed')

print('\n All dependencies ready')

Installing flask...
 flask installed
Installing flask-cors...
 flask-cors installed
Installing reportlab...
 reportlab installed
Installing Pillow...
 Pillow installed
 rouge-score already installed

 All dependencies ready


## Cell 2 — Backend: Flask API

In [6]:
APP_CODE = '''
import os, sys, json, re, io, base64, warnings, datetime
warnings.filterwarnings("ignore")

import cv2
import numpy as np
import torch
import torch.nn as nn
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from pathlib import Path
from PIL import Image
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from transformers import AutoTokenizer, AutoModel

from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer,
                                 Table, TableStyle, HRFlowable, Image as RLImage)
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_JUSTIFY

app  = Flask(__name__)
CORS(app)

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE    = 224
NUM_CLASSES = 14
MAX_SEQ_LEN = 150
EMBED_DIM   = 256
HIDDEN_DIM  = 512
NUM_LAYERS  = 2
SAVE_DIR    = Path("report_gen")
PRETRAIN    = Path("ScratchCnnModels")
VOCAB_PATH  = SAVE_DIR / "vocab.json"          # ← FIX: was missing

DISEASE_LABELS = [
    "Atelectasis","Cardiomegaly","Consolidation","Edema",
    "Enlarged Cardiomediastinum","Fracture","Lung Lesion",
    "Lung Opacity","No Finding","Pleural Effusion",
    "Pleural Other","Pneumonia","Pneumothorax","Support Devices"
]

class BioViT(nn.Module):
    def __init__(self, num_classes=14):
        super().__init__()
        self.vit      = timm.create_model("vit_base_patch16_224", pretrained=False)
        in_feats      = self.vit.head.in_features
        self.vit.head = nn.Identity()
        self.head     = nn.Sequential(nn.LayerNorm(in_feats),
                                      nn.Linear(in_feats, num_classes))
    def forward(self, x): return self.head(self.vit(x))

class ImageEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        biovit = BioViT(num_classes=NUM_CLASSES)
        ckpt   = torch.load(PRETRAIN / "BioViT.pth", map_location=DEVICE, weights_only=False)
        biovit.load_state_dict(ckpt["model_state_dict"])
        self.vit      = biovit.vit
        self.feat_dim = 768
        for p in self.vit.parameters(): p.requires_grad = False
    def forward(self, x): return self.vit(x)

class TextEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        model_name     = "dmis-lab/biobert-base-cased-v1.2"
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        try:
            self.bert = AutoModel.from_pretrained(model_name, trust_remote_code=True,
                                                   use_safetensors=True)
        except:
            self.bert = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        for p in self.bert.parameters(): p.requires_grad = False
        self.feat_dim = 768
    def forward(self, texts):
        enc = self.tokenizer(texts, padding=True, truncation=True,
                             max_length=64, return_tensors="pt").to(DEVICE)
        with torch.no_grad(): out = self.bert(**enc)
        return out.last_hidden_state[:, 0, :]

class LSTMDecoder(nn.Module):
    def __init__(self, vocab_size, pad_idx):
        super().__init__()
        encoder_dim    = 1536
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=pad_idx)
        self.fc_init_h = nn.Linear(encoder_dim, HIDDEN_DIM)
        self.fc_init_c = nn.Linear(encoder_dim, HIDDEN_DIM)
        self.lstm      = nn.LSTM(EMBED_DIM, HIDDEN_DIM, NUM_LAYERS,
                                 batch_first=True, dropout=0.3)
        self.fc_out    = nn.Linear(HIDDEN_DIM, vocab_size)
        self.dropout   = nn.Dropout(0.3)
    def forward(self, encoder_out, input_ids):
        embeds = self.dropout(self.embedding(input_ids))
        h = self.fc_init_h(encoder_out).unsqueeze(0).repeat(NUM_LAYERS, 1, 1)
        c = self.fc_init_c(encoder_out).unsqueeze(0).repeat(NUM_LAYERS, 1, 1)
        out, _ = self.lstm(embeds, (h, c))
        return self.fc_out(out)

class ViTGradCAM:
    def __init__(self, model):
        self.model       = model
        self.gradients   = None
        self.activations = None
        target_layer = model.vit.blocks[-1].norm1
        target_layer.register_forward_hook(self._save_act)
        target_layer.register_full_backward_hook(self._save_grad)
    def _save_act(self, m, i, o):  self.activations = o.detach()
    def _save_grad(self, m, gi, go): self.gradients = go[0].detach()
    def generate(self, img_tensor, class_idx):
        self.model.zero_grad()
        logits = self.model(img_tensor)
        logits[0, class_idx].backward()
        grads = self.gradients[:, 1:, :]
        acts  = self.activations[:, 1:, :]
        weights = grads.mean(dim=-1, keepdim=True)
        cam = (weights * acts).sum(dim=-1)
        cam = cam.squeeze().reshape(14, 14).cpu().numpy()
        cam = np.maximum(cam, 0)
        if cam.max() > 0: cam = cam / cam.max()
        cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
        return cam

# ── Load models at startup ────────────────────────────────────
print("Loading models...")

with open(VOCAB_PATH, encoding="utf-8") as f:
    vd = json.load(f)

if "word2idx" in vd:
    word2idx = vd["word2idx"]
    idx2word = {int(k): v for k, v in vd["idx2word"].items()}
else:
    word2idx = vd
    idx2word = {v: k for k, v in vd.items()}

VOCAB_SIZE = len(word2idx)
PAD_IDX    = word2idx["<PAD>"]
SOS_IDX    = word2idx["<SOS>"]
EOS_IDX    = word2idx["<EOS>"]

img_encoder  = ImageEncoder().to(DEVICE).eval()
text_encoder = TextEncoder().to(DEVICE).eval()
decoder      = LSTMDecoder(VOCAB_SIZE, PAD_IDX).to(DEVICE)
ckpt = torch.load(SAVE_DIR / "best_decoder.pth", map_location=DEVICE, weights_only=False)
decoder.load_state_dict(ckpt["decoder_state"])
decoder.eval()

disease_model = BioViT(NUM_CLASSES).to(DEVICE)
ckpt2 = torch.load(PRETRAIN / "BioViT.pth", map_location=DEVICE, weights_only=False)
disease_model.load_state_dict(ckpt2["model_state_dict"])
disease_model.eval()

gradcam = ViTGradCAM(disease_model)
print(f"All models loaded on {DEVICE}")

# ── Helpers ───────────────────────────────────────────────────
def preprocess_image(img_bytes):
    arr   = np.frombuffer(img_bytes, np.uint8)
    img   = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    img   = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    transform = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ])
    return transform(image=img)["image"].unsqueeze(0).to(DEVICE), img


def run_pipeline(img_tensor, raw_img, clinical_history):
    with torch.no_grad():
        img_feats   = img_encoder(img_tensor)
        text_feats  = text_encoder([clinical_history])
        encoder_out = torch.cat([img_feats, text_feats], dim=-1)

        logits = disease_model(img_tensor)
        probs  = torch.sigmoid(logits)[0].cpu().numpy()

        input_ids = torch.tensor([[SOS_IDX]], dtype=torch.long).to(DEVICE)
        generated = []
        h = decoder.fc_init_h(encoder_out).unsqueeze(0).repeat(NUM_LAYERS, 1, 1)
        c = decoder.fc_init_c(encoder_out).unsqueeze(0).repeat(NUM_LAYERS, 1, 1)
        for _ in range(MAX_SEQ_LEN):
            emb        = decoder.embedding(input_ids)
            out, (h,c) = decoder.lstm(emb, (h, c))
            logit      = decoder.fc_out(out[:, -1, :])
            next_tok   = logit.argmax(dim=-1).item()
            if next_tok == EOS_IDX: break
            generated.append(next_tok)
            input_ids = torch.tensor([[next_tok]], dtype=torch.long).to(DEVICE)

    words  = [idx2word.get(i,"") for i in generated
               if i not in [PAD_IDX, SOS_IDX, EOS_IDX]]
    report = " ".join(words)

    findings   = ""
    impression = ""
    if "impression:" in report.lower():
        parts      = re.split(r"impression:", report, flags=re.IGNORECASE, maxsplit=1)
        findings   = parts[0].replace("FINDINGS:","").replace("findings:","").strip()
        impression = parts[1].strip()
    else:
        findings   = report.strip()
        impression = "A narrative report."

    top_disease_idx = int(np.argmax(probs))
    disease_model.zero_grad()
    cam = gradcam.generate(img_tensor, top_disease_idx)
    heatmap     = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)
    raw_resized = cv2.resize(raw_img, (IMG_SIZE, IMG_SIZE))
    raw_bgr     = cv2.cvtColor(raw_resized, cv2.COLOR_RGB2BGR)
    overlay     = cv2.addWeighted(raw_bgr, 0.55, heatmap, 0.45, 0)
    overlay_b64 = base64.b64encode(
        cv2.imencode(".png", overlay)[1].tobytes()).decode()
    orig_b64    = base64.b64encode(
        cv2.imencode(".png", raw_bgr)[1].tobytes()).decode()

    diseases = [{"name": DISEASE_LABELS[i], "prob": float(probs[i])}
                for i in np.argsort(probs)[::-1]]

    return {
        "findings"   : findings,
        "impression" : impression,
        "diseases"   : diseases,
        "overlay_b64": overlay_b64,
        "orig_b64"   : orig_b64,
        "top_disease": DISEASE_LABELS[top_disease_idx],
    }


def generate_pdf(patient_name, age, sex, pid, ref_doctor,
                 clinical_history, findings, impression,
                 diseases, orig_b64, overlay_b64):
    buf = io.BytesIO()
    doc = SimpleDocTemplate(buf, pagesize=A4,
                            leftMargin=2*cm, rightMargin=2*cm,
                            topMargin=1.5*cm, bottomMargin=2*cm)
    styles    = getSampleStyleSheet()
    story     = []
    W, H      = A4
    content_w = W - 4*cm

    BLUE  = colors.HexColor("#1a4f8a")
    LBLUE = colors.HexColor("#e8f0fb")
    GRAY  = colors.HexColor("#555555")
    LGRAY = colors.HexColor("#f5f5f5")
    GREEN = colors.HexColor("#1a7a4a")
    ORANGE= colors.HexColor("#c45c00")
    RED   = colors.HexColor("#b91c1c")

    def ps(name, parent="Normal", **kw):
        return ParagraphStyle(name, parent=styles[parent], **kw)

    hosp_style  = ps("Hosp",  fontSize=22, textColor=BLUE, fontName="Helvetica-Bold",
                     alignment=TA_CENTER, spaceAfter=2)
    url_style   = ps("URL",   fontSize=9,  textColor=GRAY, alignment=TA_CENTER, spaceAfter=6)
    title_style = ps("Title", fontSize=13, textColor=BLUE, fontName="Helvetica-Bold",
                     alignment=TA_CENTER, spaceBefore=10, spaceAfter=6)
    sec_style   = ps("Sec",   fontSize=10, textColor=BLUE, fontName="Helvetica-Bold",
                     spaceBefore=10, spaceAfter=4)
    body_style  = ps("Body",  fontSize=9.5, textColor=colors.black,
                     leading=15, alignment=TA_JUSTIFY, spaceAfter=4)
    label_style = ps("Lbl",   fontSize=9,  textColor=GRAY, fontName="Helvetica-Bold")
    val_style   = ps("Val",   fontSize=9,  textColor=colors.black)

    story.append(Paragraph("SMART IMAGING CENTER", hosp_style))
    story.append(Paragraph("www.smartimaging.com", url_style))
    story.append(HRFlowable(width="100%", thickness=2, color=BLUE, spaceAfter=8))

    now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    info_data = [
        [Paragraph(f"<b>{patient_name.upper()}</b>", ps("PN", fontSize=11, fontName="Helvetica-Bold")),
         Paragraph(f"PID : {pid}", label_style),
         Paragraph("Reported on:", label_style)],
        [Paragraph(f"Age : {age} Years", val_style),
         Paragraph(f"Ref : {ref_doctor}", val_style),
         Paragraph(now, val_style)],
        [Paragraph(f"Sex : {sex}", val_style), "", ""],
    ]
    info_table = Table(info_data, colWidths=[content_w*0.38, content_w*0.35, content_w*0.27])
    info_table.setStyle(TableStyle([
        ("VALIGN",      (0,0),(-1,-1),"TOP"),
        ("TOPPADDING",  (0,0),(-1,-1),4),
        ("BOTTOMPADDING",(0,0),(-1,-1),3),
    ]))
    story.append(info_table)
    story.append(HRFlowable(width="100%", thickness=0.8, color=GRAY, spaceBefore=6, spaceAfter=10))
    story.append(Paragraph("CHEST RADIOGRAPH (PA VIEW)", title_style))
    story.append(HRFlowable(width="40%", thickness=1, color=BLUE,
                             hAlign="CENTER", spaceAfter=12))

    try:
        orig_bytes    = base64.b64decode(orig_b64)
        overlay_bytes = base64.b64decode(overlay_b64)
        orig_pil    = Image.open(io.BytesIO(orig_bytes)).convert("RGB")
        overlay_pil = Image.open(io.BytesIO(overlay_bytes)).convert("RGB")
        orig_io    = io.BytesIO(); orig_pil.save(orig_io,    "PNG"); orig_io.seek(0)
        overlay_io = io.BytesIO(); overlay_pil.save(overlay_io,"PNG"); overlay_io.seek(0)
        img_w = content_w * 0.47
        img_h = img_w
        img_table = Table(
            [[RLImage(orig_io, width=img_w, height=img_h),
              RLImage(overlay_io, width=img_w, height=img_h)]],
            colWidths=[content_w*0.5, content_w*0.5]
        )
        img_table.setStyle(TableStyle([
            ("ALIGN",         (0,0),(-1,-1),"CENTER"),
            ("VALIGN",        (0,0),(-1,-1),"MIDDLE"),
            ("LEFTPADDING",   (0,0),(-1,-1),4),
            ("RIGHTPADDING",  (0,0),(-1,-1),4),
        ]))
        story.append(img_table)
        caption_data = [[
            Paragraph("Original X-Ray",
                      ps("C1",fontSize=8,textColor=GRAY,alignment=TA_CENTER)),
            Paragraph("Saliency Activation Map",
                      ps("C2",fontSize=8,textColor=GRAY,alignment=TA_CENTER)),
        ]]
        caption_table = Table(caption_data, colWidths=[content_w*0.5, content_w*0.5])
        caption_table.setStyle(TableStyle([
            ("ALIGN",(0,0),(-1,-1),"CENTER"),
            ("TOPPADDING",(0,0),(-1,-1),2)
        ]))
        story.append(caption_table)
        story.append(Spacer(1, 10))
    except Exception as e:
        story.append(Paragraph(f"[Images unavailable: {e}]", body_style))

    story.append(Paragraph("Clinical History", sec_style))
    story.append(Paragraph(clinical_history or "Not provided.", body_style))
    story.append(Paragraph("Technique", sec_style))
    story.append(Paragraph(
        "Standard frontal projection of the chest was obtained. "
        "AI-assisted analysis was performed using BioViT deep learning model.", body_style))
    story.append(Paragraph("Comparison", sec_style))
    story.append(Paragraph("No prior examinations are available for comparison.", body_style))
    story.append(Paragraph("Findings", sec_style))
    story.append(Paragraph(findings or "No significant findings.", body_style))
    story.append(Paragraph("Impression", sec_style))
    story.append(Paragraph(impression or "A narrative report.", body_style))

    story.append(Spacer(1, 8))
    story.append(Paragraph("AI Disease Confidence Scores", sec_style))
    top5 = sorted(diseases, key=lambda x: x["prob"], reverse=True)[:5]
    dis_rows = [[
        Paragraph("Disease",    ps("DH",  fontSize=9, fontName="Helvetica-Bold", textColor=colors.white)),
        Paragraph("Confidence", ps("DH2", fontSize=9, fontName="Helvetica-Bold", textColor=colors.white)),
        Paragraph("Level",      ps("DH3", fontSize=9, fontName="Helvetica-Bold", textColor=colors.white)),
    ]]
    for d in top5:
        pct = d["prob"] * 100
        lvl = "HIGH" if pct >= 60 else "MODERATE" if pct >= 35 else "LOW"
        clr = RED if pct >= 60 else ORANGE if pct >= 35 else GREEN
        dis_rows.append([
            Paragraph(d["name"], ps("DN", fontSize=9)),
            Paragraph(f"{pct:.1f}%", ps("DP", fontSize=9, alignment=TA_CENTER)),
            Paragraph(lvl, ps("DL", fontSize=9, textColor=clr,
                              fontName="Helvetica-Bold", alignment=TA_CENTER)),
        ])
    dis_table = Table(dis_rows, colWidths=[content_w*0.55, content_w*0.22, content_w*0.23])
    dis_table.setStyle(TableStyle([
        ("BACKGROUND",    (0,0),(-1,0),  BLUE),
        ("BACKGROUND",    (0,1),(-1,1),  LBLUE),
        ("BACKGROUND",    (0,2),(-1,2),  LGRAY),
        ("BACKGROUND",    (0,3),(-1,3),  LBLUE),
        ("BACKGROUND",    (0,4),(-1,4),  LGRAY),
        ("BACKGROUND",    (0,5),(-1,5),  LBLUE),
        ("ALIGN",         (1,0),(-1,-1), "CENTER"),
        ("VALIGN",        (0,0),(-1,-1), "MIDDLE"),
        ("GRID",          (0,0),(-1,-1), 0.5, colors.white),
        ("TOPPADDING",    (0,0),(-1,-1), 5),
        ("BOTTOMPADDING", (0,0),(-1,-1), 5),
        ("LEFTPADDING",   (0,0),(-1,-1), 8),
    ]))
    story.append(dis_table)
    story.append(Spacer(1, 14))
    story.append(HRFlowable(width="100%", thickness=0.8, color=GRAY))
    story.append(Paragraph(
        "Thanks for Reference &nbsp;&nbsp;&nbsp; ****End of Report****",
        ps("End", fontSize=9, textColor=GRAY, alignment=TA_CENTER, spaceBefore=8)))
    story.append(Paragraph(
        "This report is AI-assisted. Final diagnosis must be confirmed by a qualified radiologist.",
        ps("Disc", fontSize=7.5, textColor=GRAY, alignment=TA_CENTER, spaceBefore=4)))

    doc.build(story)
    buf.seek(0)
    return buf


# ── API Routes ────────────────────────────────────────────────
@app.route("/api/analyze", methods=["POST"])
def analyze():
    try:
        if "image" not in request.files:
            return jsonify({"error": "No image uploaded"}), 400
        img_bytes = request.files["image"].read()
        history   = request.form.get("history", "")
        img_tensor, raw_img = preprocess_image(img_bytes)
        result = run_pipeline(img_tensor, raw_img, history)
        return jsonify({"status": "ok", "data": result})
    except Exception as e:
        import traceback; traceback.print_exc()
        return jsonify({"error": str(e)}), 500


@app.route("/api/download_pdf", methods=["POST"])
def download_pdf():
    try:
        body = request.get_json()
        pdf_buf = generate_pdf(
            patient_name     = body.get("patient_name",    "Patient"),
            age              = body.get("age",             "N/A"),
            sex              = body.get("sex",             "N/A"),
            pid              = body.get("pid",             "N/A"),
            ref_doctor       = body.get("ref_doctor",      "Self-Referral"),
            clinical_history = body.get("clinical_history",""),
            findings         = body.get("findings",        ""),
            impression       = body.get("impression",      ""),
            diseases         = body.get("diseases",        []),
            orig_b64         = body.get("orig_b64",        ""),
            overlay_b64      = body.get("overlay_b64",     ""),
        )
        return send_file(pdf_buf, mimetype="application/pdf",
                         as_attachment=True,
                         download_name="Radiology_Report.pdf")
    except Exception as e:
        import traceback; traceback.print_exc()
        return jsonify({"error": str(e)}), 500


@app.route("/api/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "device": str(DEVICE)})


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(APP_CODE)

print(' app.py saved!')
print('\n  Run in terminal:')
print('  > python app.py')
print('  Then open: http://localhost:5000')

 app.py saved!

  Run in terminal:
  > python app.py
  Then open: http://localhost:5000


## Cell 3 — Frontend: Beautiful Hospital-Grade UI

In [10]:
FRONTEND_HTML = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Smart Imaging Center — AI Radiology</title>
<style>
  :root {
    --blue:     #1a4f8a;
    --lblue:    #e8f0fb;
    --green:    #1a7a4a;
    --orange:   #c45c00;
    --red:      #b91c1c;
    --gray:     #555;
    --lgray:    #f5f5f7;
    --white:    #ffffff;
    --shadow:   0 4px 24px rgba(26,79,138,0.10);
    --radius:   14px;
  }
  * { box-sizing: border-box; margin: 0; padding: 0; font-family: "Segoe UI", Arial, sans-serif; }
  body { background: var(--lgray); color: #222; min-height: 100vh; }

  header {
    background: linear-gradient(135deg, var(--blue) 0%, #2563b0 100%);
    color: var(--white); padding: 18px 40px;
    display: flex; align-items: center; gap: 18px;
    box-shadow: 0 2px 12px rgba(0,0,0,0.18);
  }
  header .logo-icon {
    width: 52px; height: 52px; border-radius: 50%;
    background: rgba(255,255,255,0.18); display: flex;
    align-items: center; justify-content: center; font-size: 26px;
  }
  header .brand h1 { font-size: 22px; font-weight: 700; letter-spacing: 0.5px; }
  header .brand p  { font-size: 12px; opacity: 0.78; margin-top: 2px; }
  header .status-badge {
    margin-left: auto; background: rgba(255,255,255,0.15);
    border: 1px solid rgba(255,255,255,0.3);
    border-radius: 20px; padding: 5px 16px; font-size: 12px;
    display: flex; align-items: center; gap: 6px;
  }
  .dot { width: 8px; height: 8px; border-radius: 50%; background: #4ade80;
         animation: pulse 2s infinite; }
  @keyframes pulse { 0%,100%{opacity:1} 50%{opacity:0.4} }

  .container { max-width: 1200px; margin: 0 auto; padding: 28px 20px; }
  .grid { display: grid; grid-template-columns: 1fr 1.4fr; gap: 24px; }
  @media(max-width:900px){ .grid{ grid-template-columns:1fr; } }

  .card {
    background: var(--white); border-radius: var(--radius);
    box-shadow: var(--shadow); padding: 28px;
  }
  .card-title {
    font-size: 15px; font-weight: 700; color: var(--blue);
    margin-bottom: 18px; display: flex; align-items: center; gap: 8px;
    padding-bottom: 12px; border-bottom: 2px solid var(--lblue);
  }

  .upload-zone {
    border: 2.5px dashed #93b4e0; border-radius: 12px;
    padding: 36px 20px; text-align: center; cursor: pointer;
    transition: all 0.25s; background: var(--lblue);
    position: relative;
  }
  .upload-zone:hover, .upload-zone.drag { border-color: var(--blue); background: #dce8f7; }
  .upload-zone .icon { font-size: 44px; margin-bottom: 10px; }
  .upload-zone p { color: var(--gray); font-size: 13.5px; }
  .upload-zone input { position: absolute; inset: 0; opacity: 0; cursor: pointer; }
  #preview-wrap { margin-top: 14px; display: none; }
  #preview-wrap img { width: 100%; border-radius: 10px; max-height: 220px; object-fit: cover; }

  .form-group { margin-bottom: 14px; }
  .form-group label { display: block; font-size: 12.5px; font-weight: 600;
                      color: var(--gray); margin-bottom: 5px; }
  .form-group input, .form-group select, .form-group textarea {
    width: 100%; padding: 9px 12px; border: 1.5px solid #d0d9e8;
    border-radius: 8px; font-size: 13px; transition: border 0.2s;
    background: #fafbfd;
  }
  .form-group input:focus, .form-group select:focus, .form-group textarea:focus {
    outline: none; border-color: var(--blue); background: var(--white);
  }
  .form-row { display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 12px; }

  .btn {
    display: inline-flex; align-items: center; gap: 8px;
    padding: 11px 26px; border-radius: 9px; font-size: 14px;
    font-weight: 600; cursor: pointer; border: none; transition: all 0.2s;
  }
  .btn-primary { background: var(--blue); color: white; width: 100%; justify-content: center; }
  .btn-primary:hover { background: #153f70; transform: translateY(-1px); }
  .btn-primary:disabled { background: #93b4e0; cursor: not-allowed; transform: none; }
  .btn-pdf { background: linear-gradient(135deg,#1a7a4a,#2aab66);
              color:white; margin-top: 16px; width: 100%; justify-content: center; }
  .btn-pdf:hover { background: linear-gradient(135deg,#155e38,#239a58); transform:translateY(-1px); }

  .loading { display: none; flex-direction: column; align-items: center;
             gap: 14px; padding: 30px; }
  .spinner {
    width: 48px; height: 48px; border: 4px solid var(--lblue);
    border-top-color: var(--blue); border-radius: 50%;
    animation: spin 0.9s linear infinite;
  }
  @keyframes spin { to { transform: rotate(360deg); } }
  .loading p { color: var(--gray); font-size: 13.5px; }

  #results { display: none; }
  .images-row { display: grid; grid-template-columns: 1fr 1fr; gap: 14px; margin-bottom: 20px; }
  .img-card { border-radius: 10px; overflow: hidden; box-shadow: 0 2px 10px rgba(0,0,0,0.08); }
  .img-card img { width: 100%; height: 200px; object-fit: cover; display: block; }
  .img-label { text-align: center; font-size: 11.5px; color: var(--gray);
               padding: 7px; background: var(--lgray); font-style: italic; }

  .disease-list { display: flex; flex-direction: column; gap: 9px; margin-bottom: 20px; }
  .disease-item { display: flex; align-items: center; gap: 10px; }
  .disease-name { width: 190px; font-size: 12px; color: #333; flex-shrink: 0; }
  .bar-wrap { flex: 1; background: #e8eef5; border-radius: 20px; height: 12px; overflow: hidden; }
  .bar-fill { height: 100%; border-radius: 20px; transition: width 1s ease; }
  .bar-pct { width: 44px; font-size: 11.5px; color: var(--gray); text-align: right; flex-shrink: 0; }

  .report-section { margin-bottom: 16px; }
  .report-section h4 {
    font-size: 12px; font-weight: 700; color: var(--blue);
    text-transform: uppercase; letter-spacing: 0.8px;
    margin-bottom: 6px; padding-bottom: 5px;
    border-bottom: 1.5px solid var(--lblue);
  }
  .report-section p { font-size: 13px; color: #333; line-height: 1.7; }

  .steps { display: flex; gap: 14px; margin-bottom: 24px; flex-wrap: wrap; }
  .step { flex: 1; min-width: 100px; text-align: center; padding: 14px 10px;
          background: var(--lblue); border-radius: 10px; }
  .step .num { width: 30px; height: 30px; border-radius: 50%; background: var(--blue);
               color: white; font-size: 13px; font-weight: 700;
               display: flex; align-items: center; justify-content: center; margin: 0 auto 8px; }
  .step p { font-size: 11.5px; color: var(--gray); }

  footer { background: var(--blue); color: rgba(255,255,255,0.7);
           text-align: center; padding: 16px; font-size: 12px; margin-top: 40px; }
  footer span { color: white; font-weight: 600; }
</style>
</head>
<body>

<header>
  <div class="logo-icon">🫁</div>
  <div class="brand">
    <h1>SMART IMAGING CENTER</h1>
    <p>AI-Powered Radiology Report Generation System</p>
  </div>
  <div class="status-badge"><span class="dot"></span> AI System Online</div>
</header>

<div class="container">
  <div class="steps">
    <div class="step"><div class="num">1</div><p>Upload X-Ray</p></div>
    <div class="step"><div class="num">2</div><p>Enter Patient Info</p></div>
    <div class="step"><div class="num">3</div><p>AI Analysis</p></div>
    <div class="step"><div class="num">4</div><p>Download PDF Report</p></div>
  </div>

  <div class="grid">
    <div>
      <div class="card">
        <div class="card-title">📤 Upload & Patient Details</div>
        <div class="upload-zone" id="drop-zone">
          <input type="file" id="xray-input" accept="image/*">
          <div class="icon">🩻</div>
          <p><strong>Click or drag</strong> to upload chest X-ray</p>
          <p style="font-size:11px;margin-top:6px;opacity:0.7">PNG, JPG, JPEG supported</p>
        </div>
        <div id="preview-wrap"><img id="preview-img" src="" alt="preview"></div>

        <div style="margin-top:20px;">
          <div class="form-group">
            <label>Patient Full Name</label>
            <input type="text" id="p-name" placeholder="e.g. Raj Sharma">
          </div>
          <div class="form-row">
            <div class="form-group">
              <label>Age</label>
              <input type="text" id="p-age" placeholder="e.g. 45">
            </div>
            <div class="form-group">
              <label>Sex</label>
              <select id="p-sex">
                <option value="M">Male</option>
                <option value="F">Female</option>
                <option value="O">Other</option>
              </select>
            </div>
            <div class="form-group">
              <label>PID</label>
              <input type="text" id="p-pid" placeholder="N/A">
            </div>
          </div>
          <div class="form-group">
            <label>Referring Doctor</label>
            <input type="text" id="p-ref" placeholder="e.g. Dr. Self-Referral">
          </div>
          <div class="form-group">
            <label>Clinical History</label>
            <textarea id="p-history" rows="3"
              placeholder="e.g. Shortness of breath, chest pain, fever..."></textarea>
          </div>
        </div>

        <button class="btn btn-primary" id="analyze-btn" disabled onclick="analyzeXray()">
          🔍 Analyze X-Ray & Generate Report
        </button>
      </div>
    </div>

    <div>
      <div class="card">
        <div class="card-title">📋 AI Radiology Report</div>

        <div id="placeholder" style="text-align:center;padding:50px 20px;color:var(--gray);">
          <div style="font-size:60px;margin-bottom:16px;">🫁</div>
          <p style="font-size:15px;font-weight:600;color:var(--blue);margin-bottom:8px;">
            Ready for Analysis</p>
          <p style="font-size:13px;">Upload a chest X-ray and fill patient details<br>to generate an AI radiology report.</p>
        </div>

        <div class="loading" id="loading">
          <div class="spinner"></div>
          <p id="loading-text">Analyzing X-ray with BioViT...</p>
        </div>

        <div id="results">
          <div class="images-row">
            <div class="img-card">
              <img id="orig-img" src="" alt="Original X-Ray">
              <div class="img-label">Original X-Ray</div>
            </div>
            <div class="img-card">
              <img id="heatmap-img" src="" alt="Saliency Map">
              <div class="img-label">Saliency Activation Map</div>
            </div>
          </div>

          <div class="card-title" style="margin-bottom:12px;">🎯 Disease Confidence</div>
          <div class="disease-list" id="disease-list"></div>

          <div class="card-title" style="margin-top:20px;margin-bottom:12px;">📄 Generated Report</div>
          <div class="report-section">
            <h4>Clinical History</h4>
            <p id="r-history">—</p>
          </div>
          <div class="report-section">
            <h4>Technique</h4>
            <p>Standard frontal projection of the chest was obtained.
               AI-assisted analysis performed using BioViT deep learning model.</p>
          </div>
          <div class="report-section">
            <h4>Comparison</h4>
            <p>No prior examinations are available for comparison.</p>
          </div>
          <div class="report-section">
            <h4>Findings</h4>
            <p id="r-findings">—</p>
          </div>
          <div class="report-section">
            <h4>Impression</h4>
            <p id="r-impression">—</p>
          </div>

          <button class="btn btn-pdf" onclick="downloadPDF()">
            📥 Download Professional PDF Report
          </button>
        </div>
      </div>
    </div>
  </div>
</div>

<footer>
  <span>SMART IMAGING CENTER</span> · AI-Assisted Radiology ·
  This report is AI-generated and must be reviewed by a qualified radiologist.
</footer>

<script>
const API = 'http://localhost:5000/api';
let analysisData = null;

const dropZone   = document.getElementById('drop-zone');
const xrayInput  = document.getElementById('xray-input');
const analyzeBtn = document.getElementById('analyze-btn');
const previewWrap= document.getElementById('preview-wrap');
const previewImg = document.getElementById('preview-img');

xrayInput.addEventListener('change', e => {
  const file = e.target.files[0];
  if (!file) return;
  const reader = new FileReader();
  reader.onload = ev => {
    previewImg.src = ev.target.result;
    previewWrap.style.display = 'block';
    analyzeBtn.disabled = false;
  };
  reader.readAsDataURL(file);
});

['dragover','dragenter'].forEach(evt =>
  dropZone.addEventListener(evt, e => { e.preventDefault(); dropZone.classList.add('drag'); }));
['dragleave','drop'].forEach(evt =>
  dropZone.addEventListener(evt, e => { e.preventDefault(); dropZone.classList.remove('drag'); }));
dropZone.addEventListener('drop', e => {
  const file = e.dataTransfer.files[0];
  if (!file) return;
  xrayInput.files = e.dataTransfer.files;
  const reader = new FileReader();
  reader.onload = ev => {
    previewImg.src = ev.target.result;
    previewWrap.style.display = 'block';
    analyzeBtn.disabled = false;
  };
  reader.readAsDataURL(file);
});

async function analyzeXray() {
  const file = xrayInput.files[0];
  if (!file) { alert('Please upload an X-ray first.'); return; }

  document.getElementById('placeholder').style.display = 'none';
  document.getElementById('results').style.display     = 'none';
  document.getElementById('loading').style.display     = 'flex';
  analyzeBtn.disabled = true;

  const loadingMsgs = [
    'Analyzing X-ray with BioViT...',
    'Running Grad-CAM saliency...',
    'Encoding clinical history with BioBERT...',
    'Generating radiology report...',
    'Formatting results...'
  ];
  let msgIdx = 0;
  const msgInterval = setInterval(() => {
    document.getElementById('loading-text').textContent = loadingMsgs[msgIdx % loadingMsgs.length];
    msgIdx++;
  }, 1200);

  try {
    const formData = new FormData();
    formData.append('image',   file);
    formData.append('history', document.getElementById('p-history').value);

    const resp = await fetch(`${API}/analyze`, { method: 'POST', body: formData });
    const json = await resp.json();
    clearInterval(msgInterval);

    if (!resp.ok || json.error) throw new Error(json.error || 'Analysis failed');

    analysisData = json.data;
    renderResults(analysisData);

  } catch(err) {
    clearInterval(msgInterval);
    document.getElementById('loading').style.display     = 'none';
    document.getElementById('placeholder').style.display = 'block';
    alert('Error: ' + err.message);
    analyzeBtn.disabled = false;
  }
}

function renderResults(data) {
  document.getElementById('loading').style.display = 'none';
  document.getElementById('results').style.display = 'block';

  document.getElementById('orig-img').src    = 'data:image/png;base64,' + data.orig_b64;
  document.getElementById('heatmap-img').src = 'data:image/png;base64,' + data.overlay_b64;
  document.getElementById('r-history').textContent    = document.getElementById('p-history').value || 'Not provided.';
  document.getElementById('r-findings').textContent   = data.findings   || 'No significant findings.';
  document.getElementById('r-impression').textContent = data.impression || 'A narrative report.';

  const list = document.getElementById('disease-list');
  list.innerHTML = '';
  data.diseases.slice(0, 8).forEach(d => {
    const pct = (d.prob * 100).toFixed(1);
    const clr = d.prob >= 0.6 ? '#b91c1c' : d.prob >= 0.35 ? '#c45c00' : '#1a7a4a';
    list.innerHTML += `
      <div class="disease-item">
        <div class="disease-name">${d.name}</div>
        <div class="bar-wrap">
          <div class="bar-fill" style="width:${Math.min(d.prob*100,100)}%;background:${clr}"></div>
        </div>
        <div class="bar-pct">${pct}%</div>
      </div>`;
  });

  analyzeBtn.disabled = false;
}

async function downloadPDF() {
  if (!analysisData) { alert('Please analyze an X-ray first.'); return; }

  const payload = {
    patient_name    : document.getElementById('p-name').value    || 'Patient',
    age             : document.getElementById('p-age').value     || 'N/A',
    sex             : document.getElementById('p-sex').value     || 'N/A',
    pid             : document.getElementById('p-pid').value     || 'N/A',
    ref_doctor      : document.getElementById('p-ref').value     || 'Self-Referral',
    clinical_history: document.getElementById('p-history').value || '',
    findings        : analysisData.findings,
    impression      : analysisData.impression,
    diseases        : analysisData.diseases,
    orig_b64        : analysisData.orig_b64,
    overlay_b64     : analysisData.overlay_b64,
  };

  try {
    const resp = await fetch(`${API}/download_pdf`, {
      method : 'POST',
      headers: { 'Content-Type': 'application/json' },
      body   : JSON.stringify(payload),
    });
    if (!resp.ok) throw new Error('PDF generation failed');
    const blob = await resp.blob();
    const url  = URL.createObjectURL(blob);
    const a    = document.createElement('a');
    a.href     = url;
    a.download = `Radiology_Report_${document.getElementById('p-name').value || 'Patient'}.pdf`;
    a.click();
    URL.revokeObjectURL(url);
  } catch(err) {
    alert('PDF Error: ' + err.message);
  }
}
</script>
</body>
</html>"""

with open('index.html', 'w', encoding='utf-8') as f:
    f.write(FRONTEND_HTML)

print(' index.html saved!')
print('  1. Run: python app.py')
print('  2. Open index.html in browser')

 index.html saved!
  1. Run: python app.py
  2. Open index.html in browser


## Cell 4 — Serve Frontend from Flask (Optional)

In [ ]:
# Add this to app.py to serve the HTML directly from Flask
# So you only need to open http://localhost:5000

SERVE_CODE = '''
from flask import send_from_directory

@app.route("/")
def index():
    return send_from_directory(".", "index.html")
'''

# Append to app.py
with open('app.py', 'r') as f:
    content = f.read()

if 'send_from_directory' not in content:
    content = content.replace(
        "if __name__ == "__main__":",
        SERVE_CODE + "\nif __name__ == "__main__":"
    )
    with open('app.py', 'w') as f:
        f.write(content)
    print('✅ Flask now serves frontend at http://localhost:5000')
else:
    print('✅ Already configured')

print('\n' + '='*55)
print('  HOW TO RUN')
print('='*55)
print('  1. Open terminal in your project folder')
print('  2. Run: python app.py')
print('  3. Open browser: http://localhost:5000')
print('  4. Upload X-ray → Fill patient info → Analyze')
print('  5. Click Download PDF → Hospital report ready!')
print('='*55)

## ✅ App Complete!

### Files saved:
```
app.py       ← Flask backend (all AI models + PDF generator)
index.html   ← Hospital-grade frontend UI
```

### Run:
```bash
python app.py
# Open: http://localhost:5000
```

### Full Pipeline:
```
Upload X-Ray
      ↓
BioViT → Disease Classification (14 diseases)
      ↓
BioBERT → Encode clinical history
      ↓
LSTM Decoder → Generate findings + impression
      ↓
Grad-CAM → Saliency activation map
      ↓
Beautiful UI → Show results + disease bars
      ↓
PDF Download → Hospital-grade report (Smart Imaging Center)
```
